# 03 Basic Data Validation

This notebook checks whether the cleaned e-commerce data is ready for analysis.

Validation means checking if the data follows basic business rules.

In this notebook, I will check:
- Duplicate IDs
- Missing important IDs
- Matching IDs between related tables
- Quantity, price, revenue, and rating values
- Delivery date logic
- Support ticket date logic
- Return reason availability

In [ ]:
import pandas as pd

## Load Cleaned Datasets

In [ ]:
customers = pd.read_csv("../data/cleaned/customers_cleaned.csv")
products = pd.read_csv("../data/cleaned/products_cleaned.csv")
orders = pd.read_csv("../data/cleaned/orders_cleaned.csv")
order_items = pd.read_csv("../data/cleaned/order_items_cleaned.csv")
deliveries = pd.read_csv("../data/cleaned/deliveries_cleaned.csv")
support_tickets = pd.read_csv("../data/cleaned/support_tickets_cleaned.csv")

## Check Cleaned File Size

In [ ]:
print("Customers:", customers.shape)
print("Products:", products.shape)
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Deliveries:", deliveries.shape)
print("Support Tickets:", support_tickets.shape)

## Check Duplicate Primary IDs

Each main ID should be unique.

In [ ]:
print("Duplicate customer_id:", customers["customer_id"].duplicated().sum())
print("Duplicate product_id:", products["product_id"].duplicated().sum())
print("Duplicate order_id:", orders["order_id"].duplicated().sum())
print("Duplicate order_item_id:", order_items["order_item_id"].duplicated().sum())
print("Duplicate delivery_id:", deliveries["delivery_id"].duplicated().sum())
print("Duplicate ticket_id:", support_tickets["ticket_id"].duplicated().sum())

## Check Missing Important IDs

In [ ]:
print("Missing customer_id:", customers["customer_id"].isnull().sum())
print("Missing product_id:", products["product_id"].isnull().sum())
print("Missing order_id in orders:", orders["order_id"].isnull().sum())
print("Missing order_id in order_items:", order_items["order_id"].isnull().sum())
print("Missing order_id in deliveries:", deliveries["order_id"].isnull().sum())
print("Missing ticket_id:", support_tickets["ticket_id"].isnull().sum())

## Check Customer IDs in Orders

Every customer_id in orders should exist in customers.

In [ ]:
invalid_order_customers = orders[~orders["customer_id"].isin(customers["customer_id"])]

invalid_order_customers

## Check Order IDs in Order Items

Every order_id in order_items should exist in orders.

In [ ]:
invalid_order_items = order_items[~order_items["order_id"].isin(orders["order_id"])]

invalid_order_items

## Check Product IDs in Order Items

Every product_id in order_items should exist in products.

In [ ]:
invalid_products = order_items[~order_items["product_id"].isin(products["product_id"])]

invalid_products

## Check Order IDs in Deliveries

Every order_id in deliveries should exist in orders.

In [ ]:
invalid_delivery_orders = deliveries[~deliveries["order_id"].isin(orders["order_id"])]

invalid_delivery_orders

## Check IDs in Support Tickets

In [ ]:
invalid_ticket_orders = support_tickets[~support_tickets["order_id"].isin(orders["order_id"])]
invalid_ticket_customers = support_tickets[~support_tickets["customer_id"].isin(customers["customer_id"])]

print("Invalid order IDs in support tickets:", invalid_ticket_orders.shape[0])
print("Invalid customer IDs in support tickets:", invalid_ticket_customers.shape[0])

## Validate Quantity and Revenue

In [ ]:
print("Quantity less than or equal to 0:", order_items[order_items["quantity"] <= 0].shape[0])
print("Revenue less than 0:", order_items[order_items["revenue"] < 0].shape[0])
print("Unit price less than or equal to 0:", order_items[order_items["unit_price"] <= 0].shape[0])

## Validate Product Price and Rating

In [ ]:
print("MRP less than or equal to 0:", products[products["mrp"] <= 0].shape[0])
print("Selling price less than or equal to 0:", products[products["selling_price"] <= 0].shape[0])
print("Cost price less than or equal to 0:", products[products["cost_price"] <= 0].shape[0])

invalid_product_rating = products[
    (products["product_rating"] < 1) | 
    (products["product_rating"] > 5)
]

print("Product rating outside 1 to 5:", invalid_product_rating.shape[0])

## Validate Delivery Rating and Delay Days

Blank delivery rating is acceptable for cancelled, processing, or unrated orders.

In [ ]:
print("Negative delay days:", deliveries[deliveries["delay_days"] < 0].shape[0])

invalid_delivery_rating = deliveries[
    (deliveries["delivery_rating"].notnull()) &
    (
        (deliveries["delivery_rating"] < 1) |
        (deliveries["delivery_rating"] > 5)
    )
]

print("Delivery rating outside 1 to 5:", invalid_delivery_rating.shape[0])

## Validate Support Ticket Satisfaction Score

Blank score is acceptable for open tickets.

In [ ]:
invalid_satisfaction = support_tickets[
    (support_tickets["customer_satisfaction_score"].notnull()) &
    (
        (support_tickets["customer_satisfaction_score"] < 1) |
        (support_tickets["customer_satisfaction_score"] > 5)
    )
]

print("Satisfaction score outside 1 to 5:", invalid_satisfaction.shape[0])

## Convert Dates for Date Validation

In [ ]:
orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
deliveries["expected_delivery_date"] = pd.to_datetime(deliveries["expected_delivery_date"], errors="coerce")
deliveries["actual_delivery_date"] = pd.to_datetime(deliveries["actual_delivery_date"], errors="coerce")
support_tickets["ticket_date"] = pd.to_datetime(support_tickets["ticket_date"], errors="coerce")

## Check Delivery Date Logic

Actual delivery date should not be before order date.

In [ ]:
delivery_check = deliveries.merge(
    orders[["order_id", "order_date"]],
    on="order_id",
    how="left"
)

wrong_delivery_dates = delivery_check[
    (delivery_check["actual_delivery_date"].notnull()) &
    (delivery_check["actual_delivery_date"] < delivery_check["order_date"])
]

wrong_delivery_dates

## Check Support Ticket Date Logic

Ticket date should not be before order date.

In [ ]:
ticket_check = support_tickets.merge(
    orders[["order_id", "order_date"]],
    on="order_id",
    how="left"
)

wrong_ticket_dates = ticket_check[
    ticket_check["ticket_date"] < ticket_check["order_date"]
]

wrong_ticket_dates

## Check Returned Items Have Return Reason

In [ ]:
missing_return_reason = order_items[
    (order_items["return_status"] == "Returned") &
    (
        (order_items["return_reason"].isnull()) |
        (order_items["return_reason"] == "")
    )
]

missing_return_reason

## Final Validation Summary

In [ ]:
print("Validation Summary")
print("------------------")
print("Invalid customer IDs in orders:", invalid_order_customers.shape[0])
print("Invalid order IDs in order_items:", invalid_order_items.shape[0])
print("Invalid product IDs in order_items:", invalid_products.shape[0])
print("Invalid order IDs in deliveries:", invalid_delivery_orders.shape[0])
print("Invalid order IDs in support tickets:", invalid_ticket_orders.shape[0])
print("Invalid customer IDs in support tickets:", invalid_ticket_customers.shape[0])
print("Wrong delivery dates:", wrong_delivery_dates.shape[0])
print("Wrong ticket dates:", wrong_ticket_dates.shape[0])
print("Missing return reasons:", missing_return_reason.shape[0])

## Validation Summary

In this step, I performed basic validation checks on the cleaned e-commerce data.

The validation checks included:

1. Checking duplicate primary IDs.
2. Checking missing important IDs.
3. Checking whether customer IDs, order IDs, and product IDs match correctly across tables.
4. Checking that quantity, revenue, price, and ratings have valid values.
5. Checking that delivery dates are not before order dates.
6. Checking that support ticket dates are not before order dates.
7. Checking that returned items have return reasons.

After these checks, the cleaned data is ready for exploratory data analysis and SQL business analysis.